# G6 — Recomendação operacional e política A1 (dry-run)

Fecha o **CP técnico final**: aplica uma regra de decisão simples (limiar, sem ML sofisticado —
suficiente para o escopo da disciplina) sobre os indicadores já calculados (KPI 1 / KQI 2) e
gera um **candidato de política A1 em execução simulada** (`actuation.mode = "emulate"`),
seguindo o mesmo formato usado no `decision.json` da amostra oficial do curso.

Referência: `repo/data/code/notebooks/aula05_inferencia_decisao.ipynb` e
`repo/data/code/datasets/kpm-ue-tp-sample/decision.json`.

**Governança:** este notebook nunca aciona `AI_POLICY_COMMIT=1` — não há atuação física na RAN.


In [1]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)
import json
from datetime import datetime, timezone
import pandas as pd

FEATURES_CSV = Path("../derived/kpm_features.csv")
OUT = Path("../derived")
assert FEATURES_CSV.is_file(), "Rode 01_etl_kpm.ipynb e 02_eda_kpm.ipynb antes deste notebook"

features = pd.read_csv(FEATURES_CSV)
phase_order = pd.CategoricalDtype(["baseline", "stress", "recovery"], ordered=True)
features["phase"] = features["phase"].astype(phase_order)

RUN_ID = features["run_id"].iloc[0]
print("run_id:", RUN_ID)


run_id: ue-tp-20260804-174422


## 1. Regra de decisão (limiar, não ML)

In [2]:
# Limiar de delay aceitável: usamos o delay médio da fase `stress` (situação de referência de
# carga alta) como teto conservador — se o delay dentro da baixa carga ficar acima disso,
# a baixa carga deixa de ser "folga" e passa a ser suspeita.
limiar_delay_aceitavel = features.loc[features["phase"] == "stress", "DRB.RlcSduDelayDl"].mean()
limiar_delay_aceitavel = round(float(limiar_delay_aceitavel), 2)
print("Limiar de delay aceitável (média da fase stress, ms):", limiar_delay_aceitavel)

LIMIAR_FBC_PCT = 90.0  # mesmo espírito do limiar didático do KPI 1 (CP2)


def resumo_por_fase(df):
    fbc = df.groupby("phase", observed=True)["baixa_carga"].mean() * 100
    baixa = df[df["baixa_carga"]]
    qualidade = baixa.groupby("phase", observed=True).agg(
        n_baixa_carga=("sample_index", "count"),
        thp_ul_medio=("DRB.UEThpUl", "mean"),
        delay_medio=("DRB.RlcSduDelayDl", "mean"),
    )
    out = fbc.to_frame("fbc_pct").join(qualidade)
    out["fbc_pct"] = out["fbc_pct"].round(1)
    out["thp_ul_medio"] = out["thp_ul_medio"].round(2)
    out["delay_medio"] = out["delay_medio"].round(2)
    return out


resumo = resumo_por_fase(features)
display(resumo)


Limiar de delay aceitável (média da fase stress, ms): 161.98


,fbc_pct,n_baixa_carga,thp_ul_medio,delay_medio
phase,,,,
baseline,100.0,20,3.72,55.25
stress,1.7,1,15.16,134.79
recovery,95.0,19,3.68,81.21


### Regra aplicada por fase

```
candidatar_economia(fase) = FBC(fase) >= 90%  AND  Delay_bc(fase) <= limiar_delay_aceitavel
```

- `FBC(fase) >= 90%`: a fase precisa estar majoritariamente em baixa carga.
- `Delay_bc(fase) <= limiar_delay_aceitavel`: o delay dentro das janelas de baixa carga não
  pode superar o delay médio observado na fase de carga alta (stress) — sinal de que a baixa
  carga não está mascarando uma degradação.


In [3]:
def decidir(row):
    if pd.isna(row["delay_medio"]):
        return "sem_dados"  # nenhuma amostra em baixa carga nesta fase
    if row["fbc_pct"] >= LIMIAR_FBC_PCT and row["delay_medio"] <= limiar_delay_aceitavel:
        return "apply"
    return "do_not_apply"


resumo["decision"] = resumo.apply(decidir, axis=1)
display(resumo)

for phase, row in resumo.iterrows():
    print(f"{phase}: FBC={row['fbc_pct']}%, delay_bc={row['delay_medio']}ms -> {row['decision']}")


,fbc_pct,n_baixa_carga,thp_ul_medio,delay_medio,decision
phase,,,,,
baseline,100.0,20,3.72,55.25,apply
stress,1.7,1,15.16,134.79,do_not_apply
recovery,95.0,19,3.68,81.21,apply


baseline: FBC=100.0%, delay_bc=55.25ms -> apply
stress: FBC=1.7%, delay_bc=134.79ms -> do_not_apply
recovery: FBC=95.0%, delay_bc=81.21ms -> apply


## 2. Candidato de política A1 (dry-run)

In [4]:
fases_candidatas = resumo.index[resumo["decision"] == "apply"].astype(str).tolist()
fases_excluidas = resumo.index[resumo["decision"] != "apply"].astype(str).tolist()

evaluated_at = datetime.now(timezone.utc).isoformat()

decision_g6 = {
    "actuation": {"mode": "emulate", "real": {}},
    "evaluation": {
        "rule": "FBC(fase) >= 90% AND Delay_bc(fase) <= limiar_delay_aceitavel",
        "limiar_fbc_pct": LIMIAR_FBC_PCT,
        "limiar_delay_aceitavel_ms": limiar_delay_aceitavel,
        "por_fase": [
            {
                "phase": str(phase),
                "fbc_pct": float(row["fbc_pct"]),
                "delay_bc_ms": None if pd.isna(row["delay_medio"]) else float(row["delay_medio"]),
                "thp_bc_mbps": None if pd.isna(row["thp_ul_medio"]) else float(row["thp_ul_medio"]),
                "n_baixa_carga": int(row["n_baixa_carga"]) if not pd.isna(row["n_baixa_carga"]) else 0,
                "decision": row["decision"],
            }
            for phase, row in resumo.iterrows()
        ],
        "evaluated_at": evaluated_at,
    },
    "policy": {
        "actuation": {"mode": "emulate", "real": {}},
        "lab_context": {
            "tema": "G6 — Economia de energia (intenção simulada)",
            "kpi": "KPI 1 — Fração de tempo em baixa carga (FBC)",
            "kqi": "KQI 2 — Vazão/delay médios nas janelas de baixa carga",
            "fases_candidatas": fases_candidatas,
            "fases_excluidas": fases_excluidas,
        },
        "policy_data": {
            "energySavingIntent": {
                "scope": {"cellId": "cell-lab", "runId": RUN_ID},
                "candidatePhases": fases_candidatas,
                "excludedPhases": fases_excluidas,
                "rationale": (
                    "FBC >= 90% e delay dentro das janelas de baixa carga <= "
                    f"{limiar_delay_aceitavel} ms (referência: delay médio da fase stress)"
                ),
            }
        },
        "policy_id": f"g6-energy-saving-{RUN_ID}",
        "policytype_id": "energy-saving-intent-v1",
        "ric_id": "ric-oran",
        "service_id": "g6-energy-saving-rapp",
    },
}

decision_path = OUT / "decision_g6.json"
decision_path.write_text(json.dumps(decision_g6, indent=2, ensure_ascii=False), encoding="utf-8")
print("Artefato dry-run salvo em:", decision_path.resolve())
print(json.dumps(decision_g6, indent=2, ensure_ascii=False))


Artefato dry-run salvo em: /Users/ar/Projects/pos/Modulo9/analise-de-dados-aplicada-a-redes-de-telecomunicacoes/derived/decision_g6.json
{
  "actuation": {
    "mode": "emulate",
    "real": {}
  },
  "evaluation": {
    "rule": "FBC(fase) >= 90% AND Delay_bc(fase) <= limiar_delay_aceitavel",
    "limiar_fbc_pct": 90.0,
    "limiar_delay_aceitavel_ms": 161.98,
    "por_fase": [
      {
        "phase": "baseline",
        "fbc_pct": 100.0,
        "delay_bc_ms": 55.25,
        "thp_bc_mbps": 3.72,
        "n_baixa_carga": 20,
        "decision": "apply"
      },
      {
        "phase": "stress",
        "fbc_pct": 1.7,
        "delay_bc_ms": 134.79,
        "thp_bc_mbps": 15.16,
        "n_baixa_carga": 1,
        "decision": "do_not_apply"
      },
      {
        "phase": "recovery",
        "fbc_pct": 95.0,
        "delay_bc_ms": 81.21,
        "thp_bc_mbps": 3.68,
        "n_baixa_carga": 19,
        "decision": "apply"
      }
    ],
    "evaluated_at": "2026-08-26T21:54:00.26874

## 3. Recomendação operacional (texto)

Nas janelas com **FBC ≥ 90% e delay dentro do limiar didático** (`baseline` e `recovery`),
recomenda-se abrir uma **política A1 candidata de economia de energia em dry-run**: uma
intenção simulada de redução de uso de rádio nessas janelas, sem qualquer atuação física na
RAN (`actuation.mode = "emulate"`).

Na fase `stress` (FBC ≈ 1,7%), a recomendação é **não acionar** a política — a célula está
predominantemente em carga alta, e o único ponto em baixa carga é um outlier pontual, não uma
janela de oportunidade real.

Este resultado é consistente com a resposta já dada na "Discussão rápida (CP2)" do README:
o indicador habilita a decisão, mas ela é **condicional** ao KQI 2 confirmar que a qualidade
não está comprometida — o que se verifica nesta amostra.

**Importante:** esta é uma recomendação sobre *esta amostra específica* (100 pontos, 1
`run_id`, laboratório RFSIM) — ver seção "Limitações" no README antes de generalizar.

Próximo: usar este artefato (`derived/decision_g6.json`) e este texto para preencher a seção
"Recomendação operacional e política A1 (dry-run)" do README, e para o slide correspondente
na apresentação (Aula 06).
